Bayesian networks and word generation

Model and Questions 1-14

1. The chain rule factors a sequence into next-token distributions, allowing generation by repeatedly sampling and appending a token.

2. The first-order approximation assumes P(X_t | X_1,...,X_(t-1)) = P(X_t | X_(t-1)). Earlier history is conditionally irrelevant given the preceding token. The Bayesian network is a chain.

3. Selected conditional distributions from the six prescribed sentences are shown below. Any token absent from a row has probability zero, such as P(dog|cat).

P(next | the) = {'cat': 0.25, 'mat': 0.16666666666666666, 'rug': 0.16666666666666666, 'dog': 0.25, 'park': 0.16666666666666666}; greedy prediction = cat.

P(next | cat) = {'sat': 0.6666666666666666, 'ran': 0.3333333333333333}; greedy prediction = sat.

P(next | dog) = {'sat': 0.6666666666666666, 'ran': 0.3333333333333333}; greedy prediction = sat.

P(next | sat) = {'on': 1.0}; greedy prediction = on.

P(next | ran) = {'to': 1.0}; greedy prediction = to.

4. WordModel.counts maps context tuples to Counter rows of observed continuations.

5. WordModel.fit divides each count by its row total and stores the resulting CPT in self.tables.

6. Sampling uses Random.choices with the row probabilities as weights. Greedy decoding uses the largest probability, with alphabetic tie breaking. Sampling can select a less frequent observed token.

7. An unseen context has an empty distribution and generation stops. END terminates immediately. There is no smoothing or backoff in this experiment.

8. A sum of 0.87 indicates an incomplete or incorrectly normalized distribution. All observed rows must sum to one within floating point tolerance.

9. The most probable token reflects these six sentences and limited context; it need not match a person's expectations based on syntax, meaning, or world knowledge.

10. Sampling gives more diverse outputs. Greedy decoding repeats the same maximum-probability choices and may loop until the length limit. The saved results include 30 sampled sentences, five repeated greedy generations, and at least five sampled examples per model.

11. The second-order network gives each next token two preceding parents, indexes CPT rows by token pairs, uses two-token context, and needs more observations to estimate its larger possible context space.

12. More context can distinguish subject positions from object positions after the word 'the'. However, counts spread over more possible rows, producing sparse estimates and many unsupported contexts.

13. A behavioral specification defines the intended conditional probability, count representation, and sampling method. This lets code inspection and invariants establish whether the implementation matches the model. A vague language-model request could change the objective or architecture.

14. The Bayesian-network formulation makes conditional dependencies and independence assumptions visible, factors sequence probabilities, gives interpretable CPT entries, defines ancestral sampling, and explains the data cost of increasing context. Modern language models share the next-token objective but learn neural conditional distributions over much larger contexts and vocabularies.

Implementation decisions and validation

The second-order model begins with two START tokens and learns the first word from the initial CPT, instead of assuming every sentence starts with 'the'. A test trains on 'birds fly' and 'fish swim' to check this. The length limit prevents unbounded generation, invalid modes are rejected, and empty or unknown contexts stop cleanly. Each model has its own seeded random generator. All observed probability rows normalized exactly to 1.0 in the recorded run.

Parameter and context comparison

| Order | Observed rows | Nonzero entries | Free parameters | Unseen contexts | Zero entries in observed rows | Distinct samples / 30 |
|---|---:|---:|---:|---:|---:|---:|
| 1 | 11 | 17 | 6 | 1 | 104 | 16 |
| 2 | 15 | 19 | 4 | 129 | 146 | 6 |

The unseen-context metric uses all tuples over the 11-token output vocabulary plus START: 12 contexts for order one and 144 for order two. It intentionally includes terminal and impossible boundary contexts; it is a declared Cartesian-space sparsity measure, not the count of reachable prediction contexts. Nonzero entries count stored probabilities. Free parameters subtract one normalization constraint per observed row. Missing contexts have no estimated distribution, rather than a valid distribution whose entries all equal zero.

Generated examples

Order 1, first five samples:

- the dog sat on the dog sat on the dog ran to the dog ran to the cat ran to the rug
- the cat sat on the mat
- the mat
- the dog sat on the cat sat on the rug
- the rug

Greedy output: the cat sat on the cat sat on the cat sat on the cat sat on the cat sat on the cat sat on the cat sat on the cat

Order 2, first five samples:

- the cat sat on the mat
- the cat sat on the rug
- the dog ran to the park
- the cat sat on the mat
- the dog sat on the rug

Greedy output: the cat sat on the mat

The first-order samples can place a subject continuation after an object-position 'the', producing grammatical drift and repetitive sequences. The second-order samples from this dataset preserve the local phrase structure and produce fewer distinct complete sentences. Diversity alone is therefore not a measure of coherence. Codex helped implement counting and generation; inspection caught the reference's fixed second-order first word, and explicit start-context and normalization tests checked the replacement.

In [1]:
from collections import Counter
from itertools import product
from random import Random


CORPUS = [
    'the cat sat on the mat', 'the cat sat on the rug',
    'the dog sat on the mat', 'the dog ran to the park',
    'the cat ran to the park', 'the dog sat on the rug',
]


class WordModel:
    def __init__(self, order=1):
        if order not in (1, 2):
            raise ValueError('Order must be one or two')
        self.order = order
        self.counts = {}
        self.tables = {}
        self.vocabulary = set()

    def fit(self, sentences):
        self.counts = {}
        self.vocabulary = set()
        for sentence in sentences:
            words = sentence.lower().split() + ['<END>']
            self.vocabulary.update(words)
            sequence = ['<START>'] * self.order + words
            for index in range(self.order, len(sequence)):
                context = tuple(sequence[index - self.order:index])
                self.counts.setdefault(context, Counter())[sequence[index]] += 1
        self.tables = {context: {word: count / sum(row.values()) for word, count in row.items()}
                       for context, row in self.counts.items()}
        return self

    def distribution(self, context):
        context = tuple(context)
        if len(context) != self.order:
            raise ValueError('Context length does not match model order')
        return dict(self.tables.get(context, {}))

    def generate(self, mode='sampling', seed=0, limit=30):
        if mode not in ('sampling', 'greedy'):
            raise ValueError('Unknown generation mode')
        if limit < 0:
            raise ValueError('Length limit cannot be negative')
        random = Random(seed)
        history = ['<START>'] * self.order
        output = []
        while len(output) < limit:
            row = self.distribution(history[-self.order:])
            if not row:
                break
            words = sorted(row)
            next_word = (max(words, key=row.get) if mode == 'greedy' else
                         random.choices(words, [row[word] for word in words])[0])
            if next_word == '<END>':
                break
            output.append(next_word)
            history.append(next_word)
        return ' '.join(output)

    def statistics(self, samples):
        contexts = set(self.vocabulary) | {'<START>'}
        possible = set(product(contexts, repeat=self.order))
        return {'observed_rows': len(self.tables),
                'nonzero_entries': sum(len(row) for row in self.tables.values()),
                'free_parameters': sum(len(row) - 1 for row in self.tables.values()),
                'unseen_contexts': len(possible - self.tables.keys()),
                'context_universe': len(possible),
                'zero_entries_in_observed_rows': sum(len(self.vocabulary) - len(row) for row in self.tables.values()),
                'distinct_samples': len(set(samples)),
                'normalization_error': max((abs(sum(row.values()) - 1) for row in self.tables.values()), default=0)}


def experiment():
    result = {}
    for order in (1, 2):
        model = WordModel(order).fit(CORPUS)
        samples = [model.generate(seed=100 + index) for index in range(30)]
        result[str(order)] = {'statistics': model.statistics(samples),
                              'cpts': {' | '.join(context): row for context, row in model.tables.items()},
                              'sampled_sentences': samples,
                              'greedy_sentences': [model.generate('greedy', seed=i) for i in range(5)]}
    return result

import json
results = experiment()
print(json.dumps(results, indent=2))


{
  "1": {
    "statistics": {
      "observed_rows": 11,
      "nonzero_entries": 17,
      "free_parameters": 6,
      "unseen_contexts": 1,
      "context_universe": 12,
      "zero_entries_in_observed_rows": 104,
      "distinct_samples": 16,
      "normalization_error": 0.0
    },
    "cpts": {
      "<START>": {
        "the": 1.0
      },
      "the": {
        "cat": 0.25,
        "mat": 0.16666666666666666,
        "rug": 0.16666666666666666,
        "dog": 0.25,
        "park": 0.16666666666666666
      },
      "cat": {
        "sat": 0.6666666666666666,
        "ran": 0.3333333333333333
      },
      "sat": {
        "on": 1.0
      },
      "on": {
        "the": 1.0
      },
      "mat": {
        "<END>": 1.0
      },
      "rug": {
        "<END>": 1.0
      },
      "dog": {
        "sat": 0.6666666666666666,
        "ran": 0.3333333333333333
      },
      "ran": {
        "to": 1.0
      },
      "to": {
        "the": 1.0
      },
      "park": {
        "<END>": 1

In [2]:
for order in (1, 2):
    fitted = WordModel(order).fit(CORPUS)
    assert all(abs(sum(row.values()) - 1) < 1e-12 for row in fitted.tables.values())
    assert fitted.generate(seed=10) == fitted.generate(seed=10)
    assert fitted.generate(limit=0) == ''
    assert fitted.distribution(['unknown'] * order) == {}
assert abs(WordModel().fit(CORPUS).distribution(['the'])['cat'] - .25) < 1e-12
assert WordModel(2).fit(['birds fly', 'fish swim']).distribution(['<START>', '<START>']) == {'birds': .5, 'fish': .5}
print('Probability and generation checks passed')


Probability and generation checks passed
